# 🚲 Post-Deploy Setup — Ontology, Graph Model, Data Agents & Activators

This notebook deploys 8 items that `fabric-cicd` can't handle directly:

1. **Bicycle_Ontology_Model_New** (Ontology) — 12 entity types, 23 relationships
2. **Bicycle_Ontology_Model_New_graph** (GraphModel) — connected to bicycles_gold lakehouse
3. **Cycling-Campaign-Agent** (OperationsAgent) — campaign automation agent
4. **Bicycle Fleet Intelligence Agent** (DataAgent) — NL→SQL across lakehouse + SM + graph
5. **ontology data agent** (DataAgent) — graph-backed ontology reasoning (depends on ontology from step 1)
6. **BicycleFleet_Activator** (Reflex) — 4 alert rules (bike availability monitoring)
7. **Cycling Campaign Activator** (Reflex) — 3 alert rules (depends on ontology from step 1)

**Prerequisites**: Run `Deploy_Bicycle_RTI.ipynb` first (all 5 stages must succeed).

In [ ]:
# =============================================================
# CELL 1 — Setup & Discovery
# =============================================================
import json, requests, time, base64, os
import sempy.fabric as fabric
from notebookutils import mssparkutils

# Get workspace info
ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# Get auth token
token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Discover bicycles_gold lakehouse ID
items = fabric.list_items()
gold_lh = items[items['Display Name'] == 'bicycles_gold']
if gold_lh.empty:
    raise RuntimeError("bicycles_gold lakehouse not found! Run Deploy_Bicycle_RTI.ipynb first.")
gold_lh_id = gold_lh.iloc[0]['Id']
print(f"bicycles_gold lakehouse: {gold_lh_id}")

# Discover Bicycle Ontology Model SM ID
ont_sm = items[items['Display Name'] == 'Bicycle Ontology Model']
ont_sm_id = ont_sm.iloc[0]['Id'] if not ont_sm.empty else None
print(f"Bicycle Ontology Model SM: {ont_sm_id}")

print("\n✅ Discovery complete")

In [ ]:
# =============================================================
# CELL 2 — Deploy Ontology (Bicycle_Ontology_Model_New)
# =============================================================
# Uses the Fabric Ontology REST API to:
#  1. Create the ontology item
#  2. Load the full definition (12 entity types, 23 relationships,
#     data bindings, contextualizations) from disk
#  3. Patch workspace/lakehouse IDs for the target environment
#  4. Push the definition via updateDefinition API
#
# API: POST /v1/workspaces/{id}/ontologies
# API: POST /v1/workspaces/{id}/ontologies/{id}/updateDefinition
# Ref: https://learn.microsoft.com/en-us/rest/api/fabric/ontology/items

import re

# --- Smart path resolver: find post_deploy/definitions/ regardless of upload location ---
_candidates = [
    "/lakehouse/default/Files/post_deploy/definitions",
    "/lakehouse/default/Files/RTI-Hackathon-Demo/post_deploy/definitions",
]
DEFINITIONS_BASE = None
for _c in _candidates:
    if os.path.exists(_c):
        DEFINITIONS_BASE = _c
        break
if DEFINITIONS_BASE:
    print(f"  Definitions folder: {DEFINITIONS_BASE}")
else:
    print("  [ERROR] Cannot find post_deploy/definitions/ on lakehouse!")
    print("          Upload the post_deploy folder to the attached lakehouse under Files/")

ONTOLOGY_NAME = "Bicycle_Ontology_Model_New"
ONT_API = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/ontologies"

def load_ontology_parts(base_path):
    """Walk the ontology directory and build base64-encoded parts."""
    parts = []
    for root, _dirs, files in sorted(os.walk(base_path)):
        for fname in sorted(files):
            if fname == ".platform":
                continue
            filepath = os.path.join(root, fname)
            rel_path = os.path.relpath(filepath, base_path).replace("\\", "/")
            with open(filepath, 'rb') as f:
                raw = f.read()
            if raw.startswith(b'\xef\xbb\xbf'):
                raw = raw[3:]
            payload = base64.b64encode(raw).decode("utf-8")
            parts.append({"path": rel_path, "payload": payload, "payloadType": "InlineBase64"})
    return parts

def patch_bindings(parts, target_ws_id, target_lh_id):
    """Patch workspace/lakehouse IDs in DataBinding and Contextualization parts."""
    patched = []
    for part in parts:
        path = part["path"]
        if "DataBindings/" in path or "Contextualizations/" in path:
            try:
                content = base64.b64decode(part["payload"]).decode("utf-8")
                obj = json.loads(content)
                _patch_ids(obj, target_ws_id, target_lh_id)
                content = json.dumps(obj, indent=2, ensure_ascii=False)
                part = {**part, "payload": base64.b64encode(content.encode("utf-8")).decode("utf-8")}
            except Exception as e:
                print(f"    [WARN] Could not patch {path}: {e}")
        patched.append(part)
    return patched

def _patch_ids(obj, ws_id, lh_id):
    if isinstance(obj, dict):
        for key in list(obj.keys()):
            val = obj[key]
            lk = key.lower()
            if lk in ("workspaceid", "workspaceguid", "workspace_id"):
                obj[key] = ws_id
            elif lk in ("itemid", "lakehouseid", "artifactid", "item_id"):
                obj[key] = lh_id
            elif isinstance(val, str) and "onelake" in val.lower():
                m = re.match(r'(abfss://)([0-9a-f-]+)(@onelake[^/]*/)([0-9a-f-]+)(.*)', val, re.I)
                if m:
                    obj[key] = f"{m.group(1)}{ws_id}{m.group(3)}{lh_id}{m.group(5)}"
            elif isinstance(val, (dict, list)):
                _patch_ids(val, ws_id, lh_id)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, (dict, list)):
                _patch_ids(item, ws_id, lh_id)

def wait_lro(response, label, timeout=180):
    loc = response.headers.get("Location")
    if not loc:
        time.sleep(10)
        return True
    start = time.time()
    retry = int(response.headers.get("Retry-After", 5))
    while time.time() - start < timeout:
        time.sleep(retry)
        r = requests.get(loc, headers=headers)
        if r.status_code == 200:
            status = r.json().get("status", "")
            if status == "Succeeded":
                return True
            if status in ("Failed", "Cancelled"):
                return False
    return False

print(f"Deploying Ontology: {ONTOLOGY_NAME}")
print(f"  Workspace: {ws_id}")
print(f"  Lakehouse: {gold_lh_id}")

# Check if already exists (try items DF first, then API)
existing_ont = items[items['Display Name'] == ONTOLOGY_NAME]
if not existing_ont.empty:
    ont_id = existing_ont.iloc[0]['Id']
    print(f"  Already exists (ID: {ont_id}) — will update definition")
else:
    # Create empty ontology via dedicated API
    create_body = {
        "displayName": ONTOLOGY_NAME,
        "description": "Bicycle Fleet Ontology — 12 entity types, 23 relationships, bound to bicycles_gold",
    }
    r = requests.post(ONT_API, headers=headers, json=create_body)
    if r.status_code in (200, 201):
        ont_id = r.json().get("id", "")
        print(f"  Created: {ont_id}")
    elif r.status_code == 202:
        wait_lro(r, "create ontology")
        time.sleep(3)
        r2 = requests.get(ONT_API, headers=headers)
        ont_id = None
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        print(f"  Created (async): {ont_id}")
    elif r.status_code == 400 and "AlreadyInUse" in r.text:
        # Item was created in a previous run but items DF is stale — look it up
        print(f"  Already exists (stale cache) — looking up ID via API...")
        r2 = requests.get(ONT_API, headers=headers)
        ont_id = None
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        if ont_id:
            print(f"  Found existing: {ont_id} — will update definition")
        else:
            print(f"  [FAIL] Item exists but could not find via API")
    else:
        print(f"  [FAIL] Create: {r.status_code} {r.text[:300]}")
        ont_id = None

if ont_id:
    # Load definition from disk
    ont_dir = f"{DEFINITIONS_BASE}/Bicycle_Ontology_Model_New.Ontology" if DEFINITIONS_BASE else ""
    if not ont_dir or not os.path.exists(ont_dir):
        print(f"  [WARN] Ontology definition dir not found")
        print(f"         Upload post_deploy/ folder to lakehouse Files/ first")
    else:
        parts = load_ontology_parts(ont_dir)
        et_count = sum(1 for p in parts if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"))
        rt_count = sum(1 for p in parts if p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json"))
        db_count = sum(1 for p in parts if "DataBindings/" in p["path"])
        ctx_count = sum(1 for p in parts if "Contextualizations/" in p["path"])
        print(f"  Loaded {len(parts)} parts: {et_count} entities, {rt_count} relationships, {db_count} bindings, {ctx_count} contextualizations")

        # Patch IDs for target environment
        parts = patch_bindings(parts, ws_id, gold_lh_id)
        print(f"  Patched data bindings for target environment")

        # Push full definition
        update_url = f"{ONT_API}/{ont_id}/updateDefinition"
        body = {"definition": {"parts": parts}}
        r = requests.post(update_url, headers=headers, json=body)
        if r.status_code in (200, 201):
            print(f"  ✅ Definition pushed successfully")
        elif r.status_code == 202:
            ok = wait_lro(r, "updateDefinition")
            print(f"  ✅ Definition pushed successfully" if ok else "  [FAIL] updateDefinition failed")
        else:
            print(f"  [FAIL] updateDefinition: {r.status_code} {r.text[:300]}")

        # Verify
        verify_url = f"{ONT_API}/{ont_id}/getDefinition"
        vr = requests.post(verify_url, headers=headers)
        if vr.status_code == 200:
            vparts = vr.json().get("definition", {}).get("parts", [])
            print(f"  Verified: {len(vparts)} parts in ontology definition")
        elif vr.status_code == 202:
            print(f"  Verification pending (LRO) — check in Fabric UI")

print("\n✅ Ontology step complete")


In [ ]:
# =============================================================
# CELL 2.5 — Minimal Graph API Test (Diagnostic)
# =============================================================
# Tests whether updateDefinition works AT ALL by creating a trivial
# 1-node, 0-edge graph. If this fails, the API itself is broken.
# If this succeeds, the issue is in the Bicycle definition structure.
# =============================================================

import base64, json, time

TEST_GRAPH_NAME = "_test_minimal_graph_api"
GM_API = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/graphModels"

print("=" * 60)
print("MINIMAL GRAPH API TEST")
print("=" * 60)

# Helper: wait for LRO
def _test_wait_lro(response, timeout=120):
    loc = response.headers.get("Location")
    if not loc:
        time.sleep(5)
        return True, None
    start = time.time()
    retry = int(response.headers.get("Retry-After", 5))
    while time.time() - start < timeout:
        time.sleep(retry)
        r = requests.get(loc, headers=headers)
        if r.status_code == 200:
            body = r.json()
            status = body.get("status", "")
            if status == "Succeeded":
                return True, None
            if status in ("Failed", "Cancelled"):
                return False, json.dumps(body.get("error", body))[:500]
    return False, "Timeout"

# Step 1: Cleanup any existing test graph
print("Step 1: Cleaning up existing test graph...")
r = requests.get(GM_API, headers=headers)
for g in r.json().get("value", []):
    if g.get("displayName") == TEST_GRAPH_NAME:
        dr = requests.delete(f"{GM_API}/{g['id']}", headers=headers)
        if dr.status_code == 202:
            _test_wait_lro(dr)
        print(f"  Deleted: {g['id']}")
        time.sleep(5)

# Step 2: Create minimal graph
print(f"Step 2: Creating minimal test graph: {TEST_GRAPH_NAME}")
r = requests.post(GM_API, headers=headers, json={"displayName": TEST_GRAPH_NAME, "description": "API test"})
if r.status_code in (200, 201):
    test_id = r.json().get("id")
    print(f"  Created: {test_id}")
elif r.status_code == 202:
    _test_wait_lro(r)
    r2 = requests.get(GM_API, headers=headers)
    test_id = None
    for g in r2.json().get("value", []):
        if g.get("displayName") == TEST_GRAPH_NAME:
            test_id = g["id"]
    print(f"  Created (async): {test_id}")
else:
    test_id = None
    print(f"  FAILED: {r.status_code} {r.text[:200]}")

if test_id:
    # Step 3: Build MINIMAL definition (1 node, 0 edges, 1 data source)
    print("Step 3: Building minimal definition...")
    
    # Use the simplest possible table - onto_station is guaranteed to exist
    test_table = "onto_station"  # Known to exist in bicycles_gold
    
    # Minimal graphType: 1 node with 1 string property
    graph_type = {
        "schemaVersion": "1.0.0",
        "nodeTypes": [{
            "alias": "TestNode_nodeType",
            "labels": ["TestNode"],
            "primaryKeyProperties": ["station_id"],
            "properties": [
                {"name": "station_id", "type": "INT"},
                {"name": "station_name", "type": "STRING"}
            ]
        }],
        "edgeTypes": []  # Zero edges - simplest possible
    }
    
    # Minimal graphDefinition: 1 node table
    graph_def = {
        "schemaVersion": "1.0.0",
        "nodeTables": [{
            "id": "test_node_table",
            "nodeTypeAlias": "TestNode_nodeType",
            "dataSourceName": "TestSource",
            "propertyMappings": [
                {"propertyName": "station_id", "sourceColumn": "station_id"},
                {"propertyName": "station_name", "sourceColumn": "station_name"}
            ]
        }],
        "edgeTables": []
    }
    
    # Minimal dataSources: 1 source pointing to real table
    data_sources = {
        "dataSources": [{
            "name": "TestSource",
            "type": "DeltaTable",
            "properties": {
                "path": f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{gold_lh_id}/Tables/{test_table}"
            }
        }]
    }
    
    # Minimal styling
    styling = {
        "schemaVersion": "1.0.0",
        "modelLayout": {
            "positions": {"TestNode_nodeType": {"x": 100, "y": 100}},
            "styles": {"TestNode_nodeType": {"size": 30}},
            "pan": {"x": 0, "y": 0},
            "zoomLevel": 1
        }
    }
    
    # .platform
    platform = {
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
        "metadata": {"type": "GraphModel", "displayName": TEST_GRAPH_NAME},
        "config": {"version": "2.0", "logicalId": "00000000-0000-0000-0000-000000000000"}
    }
    
    def encode_part(path, content):
        return {"path": path, "payload": base64.b64encode(json.dumps(content, indent=2).encode()).decode(), "payloadType": "InlineBase64"}
    
    # NOTE: Only 4 parts - .platform is for Git integration, NOT for updateDefinition API
    parts = [
        encode_part("graphType.json", graph_type),
        encode_part("graphDefinition.json", graph_def),
        encode_part("dataSources.json", data_sources),
        encode_part("stylingConfiguration.json", styling)
    ]
    
    print(f"  Parts: {len(parts)} (1 node, 0 edges, 1 source)")
    
    # Step 4: Wait for graph to initialize
    print("Step 4: Waiting 20s for graph initialization...")
    time.sleep(20)
    
    # Step 5: Push definition
    print("Step 5: Pushing minimal definition via updateDefinition...")
    # Note: Don't use updateMetadata=True unless including .platform file
    url = f"{GM_API}/{test_id}/updateDefinition"
    r = requests.post(url, headers=headers, json={"definition": {"format": "json", "parts": parts}})
    
    if r.status_code in (200, 201):
        print("  ✅ MINIMAL API TEST PASSED - updateDefinition works!")
        api_test_passed = True
    elif r.status_code == 202:
        ok, err = _test_wait_lro(r)
        if ok:
            print("  ✅ MINIMAL API TEST PASSED - updateDefinition works!")
            api_test_passed = True
        else:
            print(f"  ❌ MINIMAL API TEST FAILED - LRO error: {err}")
            api_test_passed = False
    else:
        print(f"  ❌ MINIMAL API TEST FAILED: {r.status_code}")
        print(f"     {r.text}")
        api_test_passed = False
    
    # Cleanup test graph
    print("Step 6: Cleaning up test graph...")
    dr = requests.delete(f"{GM_API}/{test_id}", headers=headers)
    if dr.status_code == 202:
        _test_wait_lro(dr)
    print("  Deleted")
    
else:
    api_test_passed = False

print("=" * 60)
if api_test_passed:
    print("✅ API works! Issue is in Bicycle definition structure.")
    print("   Proceeding with full deployment (Cell 3)...")
else:
    print("❌ API is broken! updateDefinition fails even for minimal graph.")
    print("   This is a Fabric Public Preview API limitation.")
    print("   Manual graph creation in UI may be required.")
print("=" * 60)

In [ ]:
# =============================================================
# CELL 3 — Deploy Graph Model (HLS Pattern)
# =============================================================
# Uses the GraphDefinitionBuilder pattern from the HLS demo to
# dynamically generate the 5-part graph definition from ontology
# metadata stored in the lakehouse Files folder.
#
# The builder reads:
#   - EntityTypes/{id}/definition.json     (node types)
#   - EntityTypes/{id}/DataBindings/*.json (data mappings)
#   - RelationshipTypes/{id}/definition.json (edge types)
#   - RelationshipTypes/{id}/Contextualizations/*.json (edge mappings)
#
# This approach is verified to work in the HLS demo and avoids the
# issues with exported companion graph files.
# =============================================================

import math
import uuid
import base64

# Check if API test passed (from Cell 2.5)
try:
    if not api_test_passed:
        print("⚠️ SKIPPING Graph Model deployment - API test failed in Cell 2.5")
        print("   The GraphModel updateDefinition API is not working.")
        print("   Create the graph manually in the Fabric UI instead.")
        gm_success = False
        gm_id = None
        raise SystemExit("API not available")
except NameError:
    print("⚠️ Run Cell 2.5 first to test API availability")
    print("   Continuing anyway...")

GRAPH_MODEL_NAME = "Bicycle_Ontology_Model_New_graph"
ONTOLOGY_NAME = "Bicycle_Ontology_Model_New"
GM_API = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/graphModels"

# Ontology valueType → GraphModel property type
TYPE_MAP = {
    "String": "STRING",
    "BigInt": "INT",
    "Double": "FLOAT",
    "DateTime": "DATETIME",
    "Boolean": "BOOLEAN",
    "Int": "INT",
}

print(f"Deploying GraphModel: {GRAPH_MODEL_NAME}")
print(f"  Pattern: HLS GraphDefinitionBuilder (dynamic from ontology)")
gm_success = False
gm_id = None

# ── Helper: LRO poller ──
def wait_lro(response, label, timeout=300):
    loc = response.headers.get("Location")
    if not loc:
        time.sleep(10)
        return True, None
    start = time.time()
    retry = int(response.headers.get("Retry-After", 5))
    while time.time() - start < timeout:
        time.sleep(retry)
        r = requests.get(loc, headers=headers)
        if r.status_code == 200:
            body = r.json()
            status = body.get("status", "")
            if status == "Succeeded":
                return True, None
            if status in ("Failed", "Cancelled"):
                return False, json.dumps(body.get("error", body))[:500]
    return False, f"Timeout after {timeout}s"

# ── Helper: Find graph by name ──
def find_graph(name, exact=True):
    r = requests.get(GM_API, headers=headers)
    if r.status_code == 200:
        for g in r.json().get("value", []):
            dn = g.get("displayName", "")
            if (exact and dn == name) or (not exact and name in dn):
                return g["id"], dn
    return None, None

# ═══════════════════════════════════════════════════════════════
# PART A: LOAD ONTOLOGY METADATA
# ═══════════════════════════════════════════════════════════════
onto_dir = os.path.join(DEFINITIONS_BASE, f"{ONTOLOGY_NAME}.Ontology") if DEFINITIONS_BASE else ""
entity_dir = os.path.join(onto_dir, "EntityTypes")
rel_dir = os.path.join(onto_dir, "RelationshipTypes")

if not os.path.isdir(entity_dir):
    print(f"  ❌ Ontology not found: {entity_dir}")
    print(f"     Upload post_deploy/definitions/ to lakehouse Files first")
else:
    # ── Load Entities ──
    entities = {}
    for ent_name in sorted(os.listdir(entity_dir)):
        ent_path = os.path.join(entity_dir, ent_name)
        if not os.path.isdir(ent_path):
            continue
        defn_file = os.path.join(ent_path, "definition.json")
        if not os.path.exists(defn_file):
            continue
        with open(defn_file, 'r', encoding='utf-8-sig') as f:
            defn = json.load(f)
        eid = defn["id"]
        pk_prop_id = defn["entityIdParts"][0] if defn.get("entityIdParts") else None
        prop_map = {p["id"]: {"name": p["name"], "type": p["valueType"]} for p in defn.get("properties", [])}
        pk_name = prop_map[pk_prop_id]["name"] if pk_prop_id and pk_prop_id in prop_map else None
        
        bindings, source_table = {}, None
        db_dir = os.path.join(ent_path, "DataBindings")
        if os.path.isdir(db_dir):
            for f in os.listdir(db_dir):
                if not f.endswith(".json"):
                    continue
                with open(os.path.join(db_dir, f), 'r', encoding='utf-8-sig') as bf:
                    db = json.load(bf)
                cfg = db.get("dataBindingConfiguration", {})
                bindings.update({pb["targetPropertyId"]: pb["sourceColumnName"] for pb in cfg.get("propertyBindings", [])})
                source_table = cfg.get("sourceTableProperties", {}).get("sourceTableName") or source_table
        
        entities[eid] = {"name": defn["name"], "pk": pk_name, "props": prop_map, "table": source_table, "bindings": bindings}

    # ── Load Relationships ──
    relationships = {}
    if os.path.isdir(rel_dir):
        for rel_name in sorted(os.listdir(rel_dir)):
            rel_path = os.path.join(rel_dir, rel_name)
            if not os.path.isdir(rel_path):
                continue
            defn_file = os.path.join(rel_path, "definition.json")
            if not os.path.exists(defn_file):
                continue
            with open(defn_file, 'r', encoding='utf-8-sig') as f:
                defn = json.load(f)
            rid = defn["id"]
            src_id, tgt_id = defn["source"]["entityTypeId"], defn["target"]["entityTypeId"]
            
            ctx_table, src_cols, tgt_cols = None, [], []
            ctx_dir = os.path.join(rel_path, "Contextualizations")
            if os.path.isdir(ctx_dir):
                for f in os.listdir(ctx_dir):
                    if not f.endswith(".json"):
                        continue
                    with open(os.path.join(ctx_dir, f), 'r', encoding='utf-8-sig') as cf:
                        ctx = json.load(cf)
                    ctx_table = ctx.get("dataBindingTable", {}).get("sourceTableName")
                    src_cols = [sk["sourceColumnName"] for sk in ctx.get("sourceKeyRefBindings", [])]
                    tgt_cols = [tk["sourceColumnName"] for tk in ctx.get("targetKeyRefBindings", [])]
            
            relationships[rid] = {"name": defn["name"], "src": src_id, "tgt": tgt_id, "table": ctx_table, "src_cols": src_cols, "tgt_cols": tgt_cols}

    # Valid entities (those with source tables)
    valid_ids = {eid for eid, e in entities.items() if e["table"]}
    print(f"  Loaded: {len(entities)} entities ({len(valid_ids)} valid), {len(relationships)} relationships")

    # ═══════════════════════════════════════════════════════════════
    # PART B: BUILD 5-PART DEFINITION
    # ═══════════════════════════════════════════════════════════════
    
    # 1. graphType.json
    node_types = []
    for eid, e in entities.items():
        if eid not in valid_ids:
            continue
        props = [{"name": p["name"], "type": TYPE_MAP.get(p["type"], "STRING")} for pid, p in e["props"].items() if pid in e["bindings"]]
        node_types.append({"alias": f"{e['name']}_nodeType", "labels": [e["name"]], "primaryKeyProperties": [e["pk"]] if e["pk"] else [], "properties": props})
    
    edge_types = []
    for rid, r in relationships.items():
        if not r["table"] or not r["src_cols"] or not r["tgt_cols"]:
            continue
        if r["src"] not in valid_ids or r["tgt"] not in valid_ids:
            continue
        edge_types.append({
            "alias": f"{r['name']}_edgeType", "labels": [r["name"]],
            "sourceNodeType": {"alias": f"{entities[r['src']]['name']}_nodeType"},
            "destinationNodeType": {"alias": f"{entities[r['tgt']]['name']}_nodeType"},
            "properties": []
        })
    graph_type = {"schemaVersion": "1.0.0", "nodeTypes": node_types, "edgeTypes": edge_types}
    print(f"    graphType: {len(node_types)} nodes, {len(edge_types)} edges")

    # 2. graphDefinition.json
    node_tables = [{"id": f"{e['name']}_{uuid.uuid4().hex[:8]}", "nodeTypeAlias": f"{e['name']}_nodeType",
                    "dataSourceName": f"{e['table']}_Source",
                    "propertyMappings": [{"propertyName": p["name"], "sourceColumn": e["bindings"][pid]} for pid, p in e["props"].items() if pid in e["bindings"]]}
                   for eid, e in entities.items() if eid in valid_ids]
    
    edge_tables = [{"id": f"{r['name']}_{uuid.uuid4().hex[:8]}", "edgeTypeAlias": f"{r['name']}_edgeType",
                    "dataSourceName": f"{r['table']}_Source",
                    "sourceNodeKeyColumns": r["src_cols"], "destinationNodeKeyColumns": r["tgt_cols"], "propertyMappings": []}
                   for rid, r in relationships.items() if r["table"] and r["src_cols"] and r["tgt_cols"] and r["src"] in valid_ids and r["tgt"] in valid_ids]
    graph_def = {"schemaVersion": "1.0.0", "nodeTables": node_tables, "edgeTables": edge_tables}
    print(f"    graphDefinition: {len(node_tables)} nodeTables, {len(edge_tables)} edgeTables")

    # 3. dataSources.json
    tables = {e["table"] for eid, e in entities.items() if eid in valid_ids and e["table"]}
    tables |= {r["table"] for r in relationships.values() if r["table"] and r["src_cols"] and r["tgt_cols"] and r["src"] in valid_ids and r["tgt"] in valid_ids}
    data_sources = {"dataSources": [{"name": f"{t}_Source", "type": "DeltaTable",
                    "properties": {"path": f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{gold_lh_id}/Tables/{t}"}} for t in sorted(tables)]}
    print(f"    dataSources: {len(tables)} tables")

    # 4. stylingConfiguration.json
    n = len([e for eid, e in entities.items() if eid in valid_ids])
    radius = 300
    positions = {f"{e['name']}_nodeType": {"x": int(radius + radius * math.cos(2 * math.pi * i / max(n, 1))),
                 "y": int(radius + radius * math.sin(2 * math.pi * i / max(n, 1)))} for i, (eid, e) in enumerate([(eid, e) for eid, e in entities.items() if eid in valid_ids])}
    styling = {"schemaVersion": "1.0.0", "modelLayout": {"positions": positions, "styles": {k: {"size": 30} for k in positions}, "pan": {"x": 0, "y": 0}, "zoomLevel": 1}}

    # 5. .platform
    platform = {"$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
                "metadata": {"type": "GraphModel", "displayName": GRAPH_MODEL_NAME},
                "config": {"version": "2.0", "logicalId": "00000000-0000-0000-0000-000000000000"}}

    # Encode parts - NOTE: Only 4 parts, .platform is for Git integration, NOT for updateDefinition API
    def encode_part(path, content):
        return {"path": path, "payload": base64.b64encode(json.dumps(content, indent=2).encode()).decode(), "payloadType": "InlineBase64"}
    
    gm_parts = [encode_part("graphType.json", graph_type), encode_part("graphDefinition.json", graph_def),
                encode_part("dataSources.json", data_sources), encode_part("stylingConfiguration.json", styling)]
    print(f"  Total: {len(gm_parts)} definition parts")

    # ═══ DEBUG: Dump generated definition to JSON file ═══
    debug_dump = {
        "graphType": graph_type,
        "graphDefinition": graph_def,
        "dataSources": data_sources,
        "stylingConfiguration": styling,
        "platform": platform,
        "meta": {
            "entities_loaded": len(entities),
            "relationships_loaded": len(relationships),
            "valid_entity_ids": list(valid_ids),
            "node_tables": len(node_tables),
            "edge_tables": len(edge_tables),
        }
    }
    
    # Also dump the raw gm_parts for exact comparison
    from datetime import datetime
    debug_path = f"/lakehouse/default/Files/graph_debug_{datetime.now().strftime('%H%M%S')}.json"
    with open(debug_path, 'w', encoding='utf-8') as df:
        json.dump(debug_dump, df, indent=2)
    print(f"  DEBUG: Dumped definition to {debug_path}")
    
    # Print first nodeType and edgeType for quick inspection
    if graph_type.get("nodeTypes"):
        print(f"  DEBUG nodeType[0]: {json.dumps(graph_type['nodeTypes'][0], indent=2)[:500]}")
    if graph_type.get("edgeTypes"):
        print(f"  DEBUG edgeType[0]: {json.dumps(graph_type['edgeTypes'][0], indent=2)[:500]}")

    # ═══════════════════════════════════════════════════════════════
    # PART C: DEPLOY (Delete and Create Fresh - HLS Pattern)
    # ═══════════════════════════════════════════════════════════════
    # updateDefinition API is unreliable in Public Preview, so we
    # delete any existing graph and create fresh instead.
    
    print("\n  Checking for existing graph to delete...")
    
    r = requests.get(GM_API, headers=headers)
    if r.status_code == 200:
        for g in r.json().get("value", []):
            gn = g.get("displayName", "")
            if gn == GRAPH_MODEL_NAME or ONTOLOGY_NAME in gn:
                gid = g["id"]
                print(f"    Found: {gn} — deleting...")
                dr = requests.delete(f"{GM_API}/{gid}", headers=headers)
                if dr.status_code in (200, 202, 204):
                    if dr.status_code == 202:
                        wait_lro(dr, "delete")
                    print(f"    ✅ Deleted")
                    time.sleep(3)  # Brief pause after delete
                else:
                    print(f"    ⚠️ Delete returned {dr.status_code}")
    
    # Create fresh graph (with retry for name availability)
    print(f"\n  Creating fresh graph: {GRAPH_MODEL_NAME}...")
    target_id = None
    
    for create_attempt in range(6):  # Up to 6 attempts (60s total)
        r = requests.post(GM_API, headers=headers, json={"displayName": GRAPH_MODEL_NAME, "description": f"Graph for {ONTOLOGY_NAME}"})
        if r.status_code in (200, 201):
            target_id = r.json().get("id")
            print(f"  ✅ Created: {target_id}")
            break
        elif r.status_code == 202:
            ok, err = wait_lro(r, "create")
            target_id, _ = find_graph(GRAPH_MODEL_NAME)
            if target_id:
                print(f"  ✅ Created: {target_id}")
                break
            print(f"  ⚠️ Async create incomplete: {err}")
        elif "NotAvailableYet" in r.text:
            print(f"  ⏳ Name not available yet, waiting 10s... (attempt {create_attempt + 1}/6)")
            time.sleep(10)
        else:
            print(f"  ❌ Create failed: {r.status_code} {r.text[:300]}")
            break
    
    if not target_id:
        print(f"  ❌ Failed to create graph after retries")

    # Push definition to fresh graph
    if target_id:
        gm_id = target_id
        
        # Wait 30s for graph to fully initialize before pushing definition
        print(f"\n  Waiting 30s for graph initialization...")
        time.sleep(30)
        
        # Retry loop for updateDefinition
        max_retries = 3
        for attempt in range(max_retries):
            print(f"  Attempt {attempt + 1}/{max_retries}: Pushing definition...")
            # Note: Don't use updateMetadata=True unless including .platform file
            url = f"{GM_API}/{target_id}/updateDefinition"
            r = requests.post(url, headers=headers, json={"definition": {"format": "json", "parts": gm_parts}})
            
            if r.status_code in (200, 201):
                gm_success = True
                print(f"  ✅ Definition pushed successfully")
                break
            elif r.status_code == 202:
                ok, err = wait_lro(r, "updateDefinition")
                if ok:
                    gm_success = True
                    print(f"  ✅ Definition pushed (async)")
                    break
                print(f"  ⚠️ LRO failed: {err}")
            else:
                print(f"  ⚠️ Attempt {attempt + 1} failed: {r.status_code}")
                print(f"     {r.text}")
            
            if attempt < max_retries - 1:
                wait_time = 20 * (attempt + 1)
                print(f"  Retrying in {wait_time}s...")
                time.sleep(wait_time)
        
        if not gm_success:
            print(f"  ❌ updateDefinition failed after {max_retries} attempts")

        # Wait for data load
    if gm_id and gm_success:
        print(f"\n  Waiting for data load...")
        for attempt in range(60):
            r = requests.get(f"{GM_API}/{gm_id}", headers=headers)
            if r.status_code == 200:
                props = r.json().get("properties", {})
                readiness = props.get("queryReadiness", "")
                status = (props.get("lastDataLoadingStatus") or {}).get("status", "")
                if readiness == "Full" or status == "Completed":
                    print(f"  ✅ Data loaded — queryReadiness={readiness}")
                    break
                if status == "Failed":
                    print(f"  ❌ Data load failed")
                    break
                if attempt % 6 == 0:
                    print(f"    readiness={readiness}, status={status}...")
            time.sleep(5)

    # Summary
    print(f"\n  Graph Model: {GRAPH_MODEL_NAME}")
    print(f"  Status: {'✅ SUCCESS' if gm_success else '❌ FAILED'}")
    print(f"  Nodes: {len(node_types)} types, Edges: {len(edge_types)} types")

print("\n✅ Graph Model step complete")


In [ ]:
# =============================================================
# CELL 4 — Deploy Operations Agent (Cycling-Campaign-Agent)
# =============================================================
# Uses the generic items API since OperationsAgent doesn't have
# a dedicated REST API endpoint yet.
# Patches __KQL_DB__ and __WORKSPACE_ID__ placeholders in
# Configurations.json before sending.

print("Creating OperationsAgent: Cycling-Campaign-Agent ...")
oa_success = False

existing_oa = items[items['Display Name'] == 'Cycling-Campaign-Agent']
if not existing_oa.empty:
    print(f"  Already exists (ID: {existing_oa.iloc[0]['Id']}) — skipping")
    oa_success = True
else:
    # Discover KQL Database ID for the Operations Agent datasource
    kql_db_row = items[items['Display Name'] == 'bikerentaldb']
    kql_db_id = kql_db_row.iloc[0]['Id'] if not kql_db_row.empty else None
    print(f"  KQL Database (bikerentaldb): {kql_db_id or 'NOT FOUND'}")

    oa_dir = f"{DEFINITIONS_BASE}/Cycling-Campaign-Agent.OperationsAgent" if DEFINITIONS_BASE else ""
    oa_files = ['Configurations.json']

    parts = []
    if oa_dir and os.path.exists(oa_dir):
        for oa_file in oa_files:
            filepath = os.path.join(oa_dir, oa_file)
            if os.path.exists(filepath):
                with open(filepath, 'r') as f:
                    content = f.read()
                # Patch placeholders
                content = content.replace("__WORKSPACE_ID__", ws_id)
                if kql_db_id:
                    content = content.replace("__KQL_DB__", kql_db_id)
                else:
                    print("  [WARN] KQL DB not found — placeholder not patched")
                encoded = base64.b64encode(content.encode('utf-8')).decode('utf-8')
                parts.append({"path": oa_file, "payload": encoded, "payloadType": "InlineBase64"})
                print(f"  Loaded & patched: {oa_file}")
    else:
        print(f"  [WARN] Agent dir not found at {oa_dir}")

    if parts:
        # Try create with definition first
        create_body = {
            "displayName": "Cycling-Campaign-Agent",
            "type": "OperationsAgent",
            "definition": {"parts": parts},
        }
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
            headers=headers,
            json=create_body
        )
        if r.status_code in (200, 201):
            oa_id = r.json().get('id', '')
            print(f"  ✅ Created: {oa_id}")
            oa_success = True
        elif r.status_code == 202:
            ok = wait_lro(r, "create operations agent")
            if ok:
                print(f"  ✅ Created (async)")
                oa_success = True
            else:
                print(f"  ⚠️  LRO did not report success — checking workspace...")
        elif r.status_code == 500:
            # Fabric sometimes can't handle definition in create — try empty shell
            print(f"  [INFO] Create-with-definition returned 500 — trying empty shell...")
            create_body2 = {
                "displayName": "Cycling-Campaign-Agent",
                "type": "OperationsAgent",
            }
            r2 = requests.post(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                headers=headers,
                json=create_body2
            )
            if r2.status_code in (200, 201):
                oa_id = r2.json().get('id', '')
                print(f"  Created empty shell: {oa_id}")
                # Now push definition
                update_url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{oa_id}/updateDefinition"
                r3 = requests.post(update_url, headers=headers, json={"definition": {"parts": parts}})
                if r3.status_code in (200, 201):
                    print(f"  ✅ Definition pushed")
                    oa_success = True
                elif r3.status_code == 202:
                    ok = wait_lro(r3, "push OA definition")
                    if ok:
                        print(f"  ✅ Definition pushed")
                        oa_success = True
                    else:
                        print(f"  ⚠️  Definition push may not have completed — configure in Fabric UI")
                        oa_success = True  # shell exists
                else:
                    print(f"  ⚠️  Definition push returned {r3.status_code} — configure goals/instructions in Fabric UI")
                    print(f"      The agent shell exists, just needs manual configuration")
                    oa_success = True  # shell exists
            elif r2.status_code == 202:
                ok = wait_lro(r2, "create empty OA")
                if ok:
                    print(f"  Created empty shell (async) — configure in Fabric UI")
                    oa_success = True
                else:
                    print(f"  ⚠️  Empty shell LRO did not report success — checking workspace...")
            elif r2.status_code == 400 and "ItemDisplayNameNotAvailableYet" in r2.text:
                # The first 500 call actually kicked off async creation
                print(f"  [INFO] Name is reserved — the first call created the agent asynchronously")
                print(f"         Waiting 30 seconds for it to appear...")
                time.sleep(30)
            else:
                print(f"  ⚠️  Empty shell create returned {r2.status_code}")
                print(f"      Detail: {r2.text[:200]}")
        elif r.status_code == 400 and "ItemDisplayNameNotAvailableYet" in r.text:
            print(f"  [INFO] Name is reserved — agent is being created asynchronously")
            print(f"         Waiting 30 seconds for it to appear...")
            time.sleep(30)
        else:
            print(f"  ⚠️  Create returned {r.status_code}")
            print(f"      Detail: {r.text[:200]}")

        # Final check: verify the agent exists in workspace
        if not oa_success:
            time.sleep(5)
            refreshed = fabric.list_items()
            found = refreshed[refreshed['Display Name'] == 'Cycling-Campaign-Agent']
            if not found.empty:
                oa_id = found.iloc[0]['Id']
                print(f"  ✅ Agent appeared in workspace: {oa_id}")
                print(f"     KQL datasource may need manual configuration in Fabric UI (point it to bikerentaleventhouse)")
                oa_success = True
    else:
        print("  No definition files — skipping")

# ── Final status ──
if oa_success:
    print("\n✅ Operations agent step complete")
else:
    print("\n⚠️  Operations agent may still be creating — check the workspace in 2-3 minutes")
    print("   If it appears, open it in Fabric UI and configure the KQL datasource manually")

In [ ]:
# =============================================================
# CELL 5 — Deploy Bicycle Fleet Intelligence Agent (DataAgent)
# =============================================================
# Reads definition from post_deploy/definitions/ on the lakehouse,
# patches workspace, lakehouse, and SM artifact IDs, then creates
# or UPDATES the agent via the Fabric REST API.
#
# fabric-cicd may have already created this agent, but with stale
# artifact IDs (source workspace). This cell always pushes the
# patched definition so data sources resolve correctly.
# =============================================================

AGENT_NAME = "Bicycle Fleet Intelligence Agent"
AGENT_DIR  = f"{DEFINITIONS_BASE}/Bicycle Fleet Intelligence Agent.DataAgent" if DEFINITIONS_BASE else ""

# Source IDs that need patching
SRC_WORKSPACE  = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"
SRC_LAKEHOUSE  = "edb7391d-0c43-56ff-981f-fd19c6a770f7"
SRC_SM         = "340a85b8-9423-55e5-afd2-896cbe7ad542"

# The 13 gold-layer tables the agent should have access to
GOLD_TABLES = [
    "dim_station", "dim_neighbourhood", "dim_time", "dim_date", "dim_weather",
    "fact_availability", "fact_hourly_demand", "fact_rebalancing",
    "fact_weather_impact", "forecast_demand", "gold_station_snapshot",
    "gold_availability_recent", "dim_customers",
]

# Discover Bicycle RTI Analytics SM ID
sm_row = items[items['Display Name'] == 'Bicycle RTI Analytics']
sm_id  = sm_row.iloc[0]['Id'] if not sm_row.empty else None
print(f"Bicycle RTI Analytics SM: {sm_id}")

# Discover Ontology ID (needed for graph datasource)
if 'ont_id' not in dir() or not ont_id:
    ont_row = items[items['Display Name'] == 'Bicycle_Ontology_Model_New']
    ont_id = ont_row.iloc[0]['Id'] if not ont_row.empty else None
print(f"Ontology ID: {ont_id}")

def load_agent_parts(base_path, skip_graph=True):
    """Walk the agent directory and build base64-encoded parts.
    Skips .platform and optionally graph datasource (not in manifest)."""
    parts = []
    for root, _dirs, files in sorted(os.walk(base_path)):
        for fname in sorted(files):
            if fname == ".platform" or fname == "manifest.json":
                continue
            filepath = os.path.join(root, fname)
            rel = os.path.relpath(filepath, base_path).replace("\\", "/")
            # Skip graph datasource — not in original manifest.json
            if skip_graph and "graph-" in rel:
                continue
            with open(filepath, 'rb') as f:
                raw = f.read()
            if raw.startswith(b'\xef\xbb\xbf'):
                raw = raw[3:]  # strip BOM
            parts.append({"path": rel, "raw": raw, "rel": rel})
    return parts

def patch_agent_content(raw_bytes, tgt_ws, tgt_lh, tgt_sm, src_ws, src_lh, src_sm, tgt_ont=None):
    """Replace hardcoded workspace / lakehouse / SM / ontology GUIDs in JSON content."""
    try:
        text = raw_bytes.decode('utf-8')
    except UnicodeDecodeError:
        return raw_bytes  # binary file — skip
    text = text.replace(src_ws, tgt_ws)
    if tgt_lh and src_lh:
        text = text.replace(src_lh, tgt_lh)
    if tgt_sm and src_sm:
        text = text.replace(src_sm, tgt_sm)
    if tgt_ont and "__ONTOLOGY_FLEET__" in text:
        text = text.replace("__ONTOLOGY_FLEET__", tgt_ont)
    return text.encode('utf-8')

def simplify_lakehouse_elements(json_bytes):
    """Rebuild 'elements' for lakehouse datasource files with synthetic GUIDs.
    
    The exported datasource.json has a huge elements tree (~4000 lines) with GUIDs
    from the source workspace. We rebuild with clean synthetic IDs for the 13 gold tables.
    """
    import json as _json
    try:
        data = _json.loads(json_bytes)
    except (ValueError, UnicodeDecodeError):
        return json_bytes
    if data.get("type") != "lakehouse_tables":
        return json_bytes
    table_children = []
    for i, tbl in enumerate(GOLD_TABLES, start=1):
        table_children.append({
            "id": f"a1{i:06d}-0001-0001-0001-000000000001",
            "is_selected": True, "display_name": tbl,
            "type": "lakehouse_tables.table", "description": None, "children": []
        })
    data["elements"] = [{
        "id": "a1000000-sch0-grp0-0001-000000000001", "is_selected": True,
        "display_name": "Schemas", "type": "schema_grouping",
        "description": None, "children": [{
            "id": "a1000000-dbo0-sch0-0001-000000000001", "is_selected": True,
            "display_name": "dbo", "type": "lakehouse_tables.schema",
            "description": None, "children": [{
                "id": "a1000000-tbl0-grp0-0001-000000000001", "is_selected": True,
                "display_name": "Tables", "type": "table_grouping",
                "description": None, "children": table_children
            }]
        }]
    }]
    return _json.dumps(data, indent=2).encode('utf-8')


print(f"\nDeploying DataAgent: {AGENT_NAME}")
print(f"  Workspace : {ws_id}")
print(f"  Lakehouse : {gold_lh_id}")
print(f"  SM        : {sm_id}")

if not os.path.exists(AGENT_DIR):
    print(f"  [WARN] Agent dir not found at {AGENT_DIR}")
    print(f"         Upload post_deploy/ folder to lakehouse Files/ first")
    da_id = None
else:
    # Build patched definition parts
    # Skip graph files if ontology not available (invalid placeholder causes API failure)
    include_graph = bool(ont_id)
    raw_parts = load_agent_parts(AGENT_DIR, skip_graph=not include_graph)
    encoded_parts = []
    for p in raw_parts:
        patched = patch_agent_content(
            p["raw"], ws_id, gold_lh_id, sm_id,
            SRC_WORKSPACE, SRC_LAKEHOUSE, SRC_SM,
            tgt_ont=ont_id
        )
        # Rebuild lakehouse elements with clean synthetic GUIDs
        if "lakehouse-tables" in p["rel"] and p["rel"].endswith("datasource.json"):
            patched = simplify_lakehouse_elements(patched)
            print(f"    ✨ Simplified lakehouse elements: {p['rel']}")
        encoded_parts.append({
            "path": p["rel"],
            "payload": base64.b64encode(patched).decode('utf-8'),
            "payloadType": "InlineBase64"
        })

    print(f"  Patched {len(encoded_parts)} definition parts:")
    for ep in encoded_parts:
        print(f"    \u2022 {ep['path']}")
    if not include_graph:
        print(f"    ⚠️  Graph datasource excluded (ontology not yet deployed)")

    existing_da = items[items['Display Name'] == AGENT_NAME]
    if not existing_da.empty:
        # Agent exists (likely from fabric-cicd) — UPDATE definition with correct IDs
        da_id = existing_da.iloc[0]['Id']
        print(f"\n  Agent exists (ID: {da_id}) — updating definition with patched artifact IDs...")
        update_body = {"definition": {"parts": encoded_parts}}
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{da_id}/updateDefinition",
            headers=headers,
            json=update_body
        )
        if r.status_code == 200:
            print(f"  \u2705 Definition updated successfully")
        elif r.status_code == 202:
            ok = wait_lro(r, "update Bicycle Fleet Intelligence Agent definition")
            if ok:
                print(f"  \u2705 Definition updated (async)")
            else:
                print(f"  \u26a0\ufe0f  Update LRO reported failure — check agent in Fabric UI")
        else:
            print(f"  \u26a0\ufe0f  Update failed: {r.status_code} {r.text[:300]}")
            print(f"      You may need to manually re-add data sources in the Fabric UI")
    else:
        # Agent doesn't exist — CREATE it
        print(f"\n  Creating new agent...")
        create_body = {
            "displayName": AGENT_NAME,
            "type": "DataAgent",
            "definition": {"parts": encoded_parts},
        }
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
            headers=headers,
            json=create_body
        )
        if r.status_code in (200, 201):
            da_id = r.json().get('id', '')
            print(f"  \u2705 Created: {da_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create Bicycle Fleet Intelligence Agent")
            print(f"  \u2705 Created (async)" if ok else "  [FAIL] Create failed")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:500]}")

print("\n\u2705 Bicycle Fleet Intelligence Agent step complete")


In [ ]:
# =============================================================
# CELL 6 — Deploy ontology data agent (DataAgent)
# =============================================================
# Minimal agent — 3 definition files, type "ontology" datasource.
# Depends on Ontology deployed in Cell 2, and uses ont_id set there.
# Patches workspace ID + __ONTOLOGY_FLEET__ placeholder.
#
# If agent already exists (from fabric-cicd), updates the definition
# with correct ontology artifact ID.
# =============================================================

ONT_AGENT_NAME = "ontology data agent"
ONT_AGENT_DIR  = f"{DEFINITIONS_BASE}/ontology data agent.DataAgent" if DEFINITIONS_BASE else ""

# Resolve the ontology artifact ID — use the one created in Cell 2 (Ontology step)
if 'ont_id' not in dir() or not ont_id:
    # Try to discover from workspace items
    ont_row = items[items['Display Name'] == 'Bicycle_Ontology_Model_New']
    ont_id = ont_row.iloc[0]['Id'] if not ont_row.empty else None
    print(f"Discovered Ontology ID: {ont_id}")
else:
    print(f"Using Ontology ID from Cell 2: {ont_id}")

def patch_ontology_agent_content(raw_bytes, tgt_ws, ontology_id, src_ws):
    """Replace workspace ID and __ONTOLOGY_FLEET__ placeholder."""
    try:
        text = raw_bytes.decode('utf-8')
    except UnicodeDecodeError:
        return raw_bytes
    text = text.replace(src_ws, tgt_ws)
    if ontology_id:
        text = text.replace("__ONTOLOGY_FLEET__", ontology_id)
    return text.encode('utf-8')

print(f"\nDeploying DataAgent: {ONT_AGENT_NAME}")
print(f"  Workspace  : {ws_id}")
print(f"  Ontology ID: {ont_id}")

if not ont_id:
    print("  [WARN] Ontology not deployed yet — run Cell 2 (Ontology) first!")
    oa2_id = None
elif not os.path.exists(ONT_AGENT_DIR):
    print(f"  [WARN] Agent dir not found at {ONT_AGENT_DIR}")
    oa2_id = None
else:
    # Build patched definition parts
    raw_parts = load_agent_parts(ONT_AGENT_DIR, skip_graph=False)
    encoded_parts = []
    for p in raw_parts:
        patched = patch_ontology_agent_content(
            p["raw"], ws_id, ont_id, SRC_WORKSPACE
        )
        encoded_parts.append({
            "path": p["rel"],
            "payload": base64.b64encode(patched).decode('utf-8'),
            "payloadType": "InlineBase64"
        })

    print(f"  Patched {len(encoded_parts)} definition parts:")
    for ep in encoded_parts:
        print(f"    \u2022 {ep['path']}")

    existing_oa2 = items[items['Display Name'] == ONT_AGENT_NAME]
    if not existing_oa2.empty:
        # Agent exists (likely from fabric-cicd) — UPDATE definition with correct ontology ID
        oa2_id = existing_oa2.iloc[0]['Id']
        print(f"\n  Agent exists (ID: {oa2_id}) — updating definition with patched ontology ID...")
        update_body = {"definition": {"parts": encoded_parts}}
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{oa2_id}/updateDefinition",
            headers=headers,
            json=update_body
        )
        if r.status_code == 200:
            print(f"  \u2705 Definition updated successfully")
        elif r.status_code == 202:
            ok = wait_lro(r, "update ontology data agent definition")
            if ok:
                print(f"  \u2705 Definition updated (async)")
            else:
                print(f"  \u26a0\ufe0f  Update LRO reported failure — check agent in Fabric UI")
        else:
            print(f"  \u26a0\ufe0f  Update failed: {r.status_code} {r.text[:300]}")
            print(f"      You may need to manually re-add the ontology data source in the Fabric UI")
    else:
        # Agent doesn't exist — CREATE it
        print(f"\n  Creating new agent...")
        create_body = {
            "displayName": ONT_AGENT_NAME,
            "type": "DataAgent",
            "definition": {"parts": encoded_parts},
        }
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
            headers=headers,
            json=create_body
        )
        if r.status_code in (200, 201):
            oa2_id = r.json().get('id', '')
            print(f"  \u2705 Created: {oa2_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create ontology data agent")
            print(f"  \u2705 Created (async)" if ok else "  [FAIL] Create failed")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:500]}")

print("\n\u2705 ontology data agent step complete")


In [ ]:
# =============================================================
# CELL 7 — Deploy BicycleFleet Activator (Reflex)
# =============================================================
# The BicycleFleet_Activator monitors bike availability in real-time.
# fabric-cicd doesn't support the Reflex item type, so we deploy via
# the Items REST API instead.
#
# 4 Rules:
#   - Empty Station (Teams) — triggers when No_Bikes == 0
#   - Full Station (Teams) — triggers when No_Empty_Docks == 0
#   - Low Availability (Email) — triggers when No_Bikes < 3
#   - High Demand (Teams) — triggers when No_Empty_Docks < 3
#
# __ALERT_RECIPIENT_EMAIL__ is replaced with the deploying user's email.
# Source workspace GUID is replaced with the target workspace.

import base64

BF_ACTIVATOR_NAME = "BicycleFleet_Activator"
SOURCE_WORKSPACE_ID = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"

# Auto-detect user email from token
def _get_email():
    try:
        tok = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
        payload = tok.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = json.loads(base64.b64decode(payload))
        return claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username", "")
    except Exception:
        return ""

bf_email = _get_email()
print(f"📧 Alert email: {bf_email or '(not detected — placeholder will remain)'}")

# Check if already exists
existing_bf = items[items['Display Name'] == BF_ACTIVATOR_NAME] if 'Display Name' in items.columns else pd.DataFrame()
if not existing_bf.empty:
    print(f"ℹ️  '{BF_ACTIVATOR_NAME}' already exists — skipping")
else:
    # Find the Reflex definition in lakehouse
    bf_dir = None
    for candidate in [
        f"/lakehouse/default/Files/workspace/{BF_ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo/workspace/{BF_ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace/{BF_ACTIVATOR_NAME}.Reflex",
    ]:
        if os.path.isdir(candidate):
            bf_dir = candidate
            break

    if not bf_dir:
        print(f"⚠️ Could not find '{BF_ACTIVATOR_NAME}.Reflex' folder in lakehouse")
        print("   Expected in: /lakehouse/default/Files/workspace/")
    else:
        entities_path = os.path.join(bf_dir, "ReflexEntities.json")
        if os.path.exists(entities_path):
            with open(entities_path, "r", encoding="utf-8") as f:
                bf_content = f.read()

            # Patch placeholders
            patches = 0
            if ws_id != SOURCE_WORKSPACE_ID and SOURCE_WORKSPACE_ID in bf_content:
                bf_content = bf_content.replace(SOURCE_WORKSPACE_ID, ws_id)
                patches += 1
                print(f"   🔄 Replaced source workspace GUID → {ws_id}")

            if bf_email and "__ALERT_RECIPIENT_EMAIL__" in bf_content:
                bf_content = bf_content.replace("__ALERT_RECIPIENT_EMAIL__", bf_email)
                patches += 1
                print(f"   📧 Replaced __ALERT_RECIPIENT_EMAIL__ → {bf_email}")

            print(f"   Applied {patches} patch(es)")

            # Encode and create via REST API
            bf_b64 = base64.b64encode(bf_content.encode("utf-8")).decode("utf-8")
            create_body = {
                "displayName": BF_ACTIVATOR_NAME,
                "type": "Reflex",
                "definition": {
                    "parts": [
                        {
                            "path": "ReflexEntities.json",
                            "payload": bf_b64,
                            "payloadType": "InlineBase64"
                        }
                    ]
                }
            }

            r = requests.post(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                headers=headers,
                json=create_body
            )

            if r.status_code in (200, 201):
                bf_id = r.json().get("id", "")
                print(f"✅ Created '{BF_ACTIVATOR_NAME}': {bf_id}")
            elif r.status_code == 202:
                ok = wait_lro(r, f"create {BF_ACTIVATOR_NAME}")
                print(f"✅ Created '{BF_ACTIVATOR_NAME}' (async)" if ok else f"❌ Failed to create '{BF_ACTIVATOR_NAME}'")
            else:
                print(f"❌ Failed to create '{BF_ACTIVATOR_NAME}': HTTP {r.status_code}")
                print(f"   {r.text[:500]}")
        else:
            print(f"⚠️ ReflexEntities.json not found in {bf_dir}")

print("\n✅ BicycleFleet Activator step complete")

In [ ]:
# =============================================================
# CELL 8 — Deploy Cycling Campaign Activator (Reflex)
# =============================================================
# The Cycling Campaign Activator depends on:
#   1. The Ontology (created in Cell 2) — __ONTOLOGY_NEW__ placeholder
#   2. Power Automate flow (per-user) — kept as-is, user connects manually
#   3. User's email — __ALERT_RECIPIENT_EMAIL__ placeholder
#
# This cell:
#   a. Reads the ReflexEntities.json from the lakehouse
#   b. Replaces __ONTOLOGY_NEW__ with the actual ontology item ID
#   c. Replaces source workspace GUID with target
#   d. Replaces __ALERT_RECIPIENT_EMAIL__ with deploying user's email
#   e. Creates the Activator via the Items REST API
#
# NOTE: The Power Automate flow action will need manual connection
# after deployment. See README for Power Automate Developer Plan setup.

import base64

ACTIVATOR_NAME = "Cycling Campaign Activator"
SOURCE_WORKSPACE_ID = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"

# Auto-detect user email from token
def _get_alert_email():
    try:
        tok = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
        payload = tok.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = json.loads(base64.b64decode(payload))
        return claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username", "")
    except Exception:
        return ""

alert_email = _get_alert_email()
print(f"📧 Alert email: {alert_email or '(not detected — placeholder will remain)'}")

# Find the deployed ontology ID (created in Cell 2)
ontology_id = None
resp = requests.get(f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                    headers=headers)
if resp.status_code == 200:
    for item in resp.json().get("value", []):
        if item.get("displayName") == "Bicycle_Ontology_Model_New" and item.get("type") == "Ontology":
            ontology_id = item["id"]
            break

if ontology_id:
    print(f"✅ Found Ontology: {ontology_id}")
else:
    print("⚠️ Ontology 'Bicycle_Ontology_Model_New' not found — run Cell 2 first")
    print("   __ONTOLOGY_NEW__ placeholder will remain (Activator won't connect to ontology)")

# Check if already exists
existing = items[items['Display Name'] == ACTIVATOR_NAME] if 'Display Name' in items.columns else pd.DataFrame()
if not existing.empty:
    print(f"ℹ️  '{ACTIVATOR_NAME}' already exists — skipping")
else:
    # Read the Reflex definition from lakehouse
    reflex_dir = None
    for candidate in [
        f"/lakehouse/default/Files/workspace/{ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo/workspace/{ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace/{ACTIVATOR_NAME}.Reflex",
    ]:
        if os.path.isdir(candidate):
            reflex_dir = candidate
            break

    if not reflex_dir:
        print(f"⚠️ Could not find '{ACTIVATOR_NAME}.Reflex' folder in lakehouse")
        print("   Expected in: /lakehouse/default/Files/workspace/")
    else:
        # Read ReflexEntities.json
        entities_path = os.path.join(reflex_dir, "ReflexEntities.json")
        if os.path.exists(entities_path):
            with open(entities_path, "r", encoding="utf-8") as f:
                entities_content = f.read()

            # Patch placeholders
            patches = 0
            if ontology_id and "__ONTOLOGY_NEW__" in entities_content:
                entities_content = entities_content.replace("__ONTOLOGY_NEW__", ontology_id)
                patches += 1
                print(f"   🔄 Replaced __ONTOLOGY_NEW__ → {ontology_id}")

            if ws_id != SOURCE_WORKSPACE_ID and SOURCE_WORKSPACE_ID in entities_content:
                entities_content = entities_content.replace(SOURCE_WORKSPACE_ID, ws_id)
                patches += 1
                print(f"   🔄 Replaced source workspace GUID → {ws_id}")

            if alert_email and "__ALERT_RECIPIENT_EMAIL__" in entities_content:
                entities_content = entities_content.replace("__ALERT_RECIPIENT_EMAIL__", alert_email)
                patches += 1
                print(f"   📧 Replaced __ALERT_RECIPIENT_EMAIL__ → {alert_email}")

            print(f"   Applied {patches} patch(es)")

            # Encode for the API
            entities_b64 = base64.b64encode(entities_content.encode("utf-8")).decode("utf-8")

            # Create the Reflex item
            create_body = {
                "displayName": ACTIVATOR_NAME,
                "type": "Reflex",
                "definition": {
                    "parts": [
                        {
                            "path": "ReflexEntities.json",
                            "payload": entities_b64,
                            "payloadType": "InlineBase64"
                        }
                    ]
                }
            }

            r = requests.post(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                headers=headers,
                json=create_body
            )

            if r.status_code in (200, 201):
                act_id = r.json().get("id", "")
                print(f"✅ Created '{ACTIVATOR_NAME}': {act_id}")
            elif r.status_code == 202:
                ok = wait_lro(r, f"create {ACTIVATOR_NAME}")
                print(f"✅ Created '{ACTIVATOR_NAME}' (async)" if ok else f"❌ Failed to create '{ACTIVATOR_NAME}'")
            else:
                print(f"❌ Failed to create '{ACTIVATOR_NAME}': HTTP {r.status_code}")
                print(f"   {r.text[:500]}")
        else:
            print(f"⚠️ ReflexEntities.json not found in {reflex_dir}")

print("\n✅ Cycling Campaign Activator step complete")
print("   ⚠️ Power Automate flow must be connected manually in Fabric UI")
print("   See README → 'Power Automate Setup' section")

In [ ]:
# =============================================================
# CELL 8b — Patch KQL Dashboard Data Sources
# =============================================================
# fabric-cicd deployed the KQL Dashboard but cannot patch the
# Eventhouse connection info.  The dashboard JSON contains two
# placeholders in its dataSources section:
#   __EVENTHOUSE_QUERY_URI__  →  Eventhouse queryServiceUri
#   __KQL_DB_ID__             →  KQL Database item GUID
#
# This cell discovers the real values and pushes a patched
# definition via the Items updateDefinition API.
# =============================================================

DASHBOARD_NAME = "Bicycle Fleet Intelligence \u2014 Live Operations"
EVENTHOUSE_NAME = "bikerentaleventhouse"
KQL_DB_NAME = "bikerentaldb"

print(f"Patching KQL Dashboard: {DASHBOARD_NAME}")
dash_success = False

# ── Step 1: Find the dashboard ──
dash_row = items[items['Display Name'] == DASHBOARD_NAME]
if dash_row.empty:
    # Try partial match
    dash_row = items[items['Display Name'].str.contains('Live Operations', case=False, na=False)]
if dash_row.empty:
    print("  ❌ Dashboard not found in workspace")
else:
    dash_id = dash_row.iloc[0]['Id']
    dash_type = dash_row.iloc[0].get('Type', 'KQLDashboard')
    print(f"  Dashboard ID: {dash_id}  (type: {dash_type})")

    # ── Step 2: Discover Eventhouse query URI ──
    query_uri = None
    eh_row = items[items['Display Name'] == EVENTHOUSE_NAME]
    if not eh_row.empty:
        eh_id = eh_row.iloc[0]['Id']
        print(f"  Eventhouse ID: {eh_id}")
        # Get properties for queryServiceUri
        eh_url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/eventhouses/{eh_id}"
        r = requests.get(eh_url, headers=headers)
        if r.status_code == 200:
            props = r.json().get("properties", {})
            query_uri = props.get("queryServiceUri") or props.get("uri")
            print(f"  Query URI: {query_uri}")
        else:
            print(f"  ⚠️  Could not get Eventhouse properties: {r.status_code}")
    else:
        print(f"  ⚠️  Eventhouse '{EVENTHOUSE_NAME}' not found")

    # ── Step 3: Discover KQL Database ID ──
    kql_db_id = None
    kql_row = items[(items['Display Name'] == KQL_DB_NAME) | (items['Display Name'] == EVENTHOUSE_NAME)]
    # KQL databases may show as the eventhouse name or the db name
    for _, row in items.iterrows():
        if row['Display Name'] in (KQL_DB_NAME, EVENTHOUSE_NAME) and 'KQL' in str(row.get('Type', '')):
            kql_db_id = row['Id']
            break
    if not kql_db_id:
        # Try the KQL databases API
        kql_url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items?type=KQLDatabase"
        r = requests.get(kql_url, headers=headers)
        if r.status_code == 200:
            for item in r.json().get("value", []):
                if item["displayName"] == KQL_DB_NAME:
                    kql_db_id = item["id"]
                    break
    if kql_db_id:
        print(f"  KQL DB ID: {kql_db_id}")
    else:
        print(f"  ⚠️  KQL Database '{KQL_DB_NAME}' not found")

    # ── Step 4: Get current definition ──
    if query_uri or kql_db_id:
        print(f"\n  Getting current dashboard definition...")
        get_def_url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{dash_id}/getDefinition"
        r = requests.post(get_def_url, headers=headers)

        def_parts = None
        if r.status_code == 200:
            def_parts = r.json().get("definition", {}).get("parts", [])
        elif r.status_code == 202:
            # Poll the LRO
            loc = r.headers.get("Location")
            if loc:
                for _ in range(30):
                    time.sleep(3)
                    r2 = requests.get(loc, headers=headers)
                    if r2.status_code == 200:
                        status = r2.json().get("status", "")
                        if status == "Succeeded":
                            # Get the result
                            result_url = loc.rstrip("/") + "/result"
                            r3 = requests.get(result_url, headers=headers)
                            if r3.status_code == 200:
                                def_parts = r3.json().get("definition", {}).get("parts", [])
                            break
                        elif status in ("Failed", "Cancelled"):
                            print(f"  ❌ getDefinition LRO {status}")
                            break
        else:
            print(f"  ❌ getDefinition: {r.status_code} {r.text[:200]}")

        if def_parts:
            print(f"  Got {len(def_parts)} definition parts")

            # ── Step 5: Patch placeholders ──
            patched = False
            for part in def_parts:
                if part["path"] == "RealTimeDashboard.json":
                    raw = base64.b64decode(part["payload"]).decode("utf-8")

                    # Check for placeholders
                    has_uri_placeholder = "__EVENTHOUSE_QUERY_URI__" in raw
                    has_db_placeholder = "__KQL_DB_ID__" in raw

                    if not has_uri_placeholder and not has_db_placeholder:
                        print("  ✅ No placeholders found — dashboard already patched!")
                        dash_success = True
                        break

                    print(f"  Placeholders found: URI={has_uri_placeholder}, DB_ID={has_db_placeholder}")

                    if query_uri and has_uri_placeholder:
                        raw = raw.replace("__EVENTHOUSE_QUERY_URI__", query_uri)
                        print(f"    Patched __EVENTHOUSE_QUERY_URI__ → {query_uri}")

                    if kql_db_id and has_db_placeholder:
                        raw = raw.replace("__KQL_DB_ID__", kql_db_id)
                        print(f"    Patched __KQL_DB_ID__ → {kql_db_id}")

                    # Re-encode
                    part["payload"] = base64.b64encode(raw.encode("utf-8")).decode("utf-8")
                    patched = True

            if patched:
                # ── Step 6: Push updated definition ──
                print(f"\n  Pushing patched definition...")
                update_url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{dash_id}/updateDefinition"
                body = {"definition": {"parts": def_parts}}
                r = requests.post(update_url, headers=headers, json=body)
                if r.status_code in (200, 201):
                    print(f"  ✅ Dashboard patched successfully!")
                    dash_success = True
                elif r.status_code == 202:
                    ok = wait_lro(r, "updateDefinition", timeout=60)
                    if ok:
                        print(f"  ✅ Dashboard patched successfully!")
                        dash_success = True
                    else:
                        print(f"  ⚠️  Dashboard update LRO did not confirm success")
                else:
                    print(f"  ❌ updateDefinition: {r.status_code} {r.text[:300]}")
        else:
            print(f"  ❌ Could not retrieve dashboard definition")
    else:
        print(f"\n  ⚠️  Neither Eventhouse URI nor KQL DB ID found — cannot patch")
        print(f"      Open the dashboard in Fabric UI and configure the data source manually:")
        print(f"      1. Open '{DASHBOARD_NAME}'")
        print(f"      2. Click the data source icon → Edit")
        print(f"      3. Connect to '{EVENTHOUSE_NAME}' → '{KQL_DB_NAME}'")

# ── Final status ──
print()
if dash_success:
    print(f"✅ KQL Dashboard patched: {DASHBOARD_NAME}")
    print(f"   Refresh the dashboard in your browser to see the tiles")
else:
    print(f"⚠️  Dashboard may need manual data source configuration:")
    print(f"   1. Open '{DASHBOARD_NAME}' in Fabric UI")
    print(f"   2. Click the data source icon → Edit")
    print(f"   3. Connect to '{EVENTHOUSE_NAME}' → '{KQL_DB_NAME}'")


## ✅ Post-Deploy Complete!

### What was deployed (8 items)

| Item | API | Status |
|------|-----|--------|
| **Bicycle_Ontology_Model_New** | `POST /ontologies` + `updateDefinition` | 12 entity types, 23 relationships, data bindings |
| **Bicycle_Ontology_Model_New_graph** | `POST /graphModels` + `updateDefinition` | 4-part definition (graphType, dataSources, graphDefinition, styling) |
| **Cycling-Campaign-Agent** | `POST /items` | Operations agent with campaign instructions |
| **Bicycle Fleet Intelligence Agent** | `POST /items` (DataAgent) | NL→SQL across lakehouse + SM (10 definition parts) |
| **ontology data agent** | `POST /items` (DataAgent) | Graph-backed ontology reasoning (3 definition parts) |
| **BicycleFleet_Activator** | `POST /items` (Reflex) | 4 rules: Empty Station, Full Station, Low Availability, High Demand |
| **Cycling Campaign Activator** | `POST /items` (Reflex) | 3 rules: High Demand Forecast, Station Critical, Cycling Campaign (Power Automate) |

### Remaining Manual Steps

1. **Refresh Graph Model**: Open `Bicycle_Ontology_Model_New_graph` in Fabric UI → click **Refresh now**
   - API refresh returns `InvalidJobType` for ontology-managed graphs
   - This is a known limitation — only the UI button works
2. **Connect Power Automate flow**: Open `Cycling Campaign Activator` → open the "Cycling Campaign" rule → link to your Power Automate flow
   - Requires Power Automate Developer Plan (free) — see README
3. **Connect BicycleFleet_Activator to eventstream**: Open `RTIbikeRental` eventstream → add an Activator destination → select `BicycleFleet_Activator`
4. **Start Eventstreams**: Open `RTIbikeRental` and `RTI-WeatherDemo` → verify they are streaming
5. **Run the Pipeline**: Open `PL_BicycleRTI_Medallion` → Run (first load ~15-25 min)
6. **Test Data Agents**: Open `Bicycle Fleet Intelligence Agent` → ask: *"What are the top 5 busiest stations?"* | Open `ontology data agent` → ask: *"Show me stations in Sands End"*

In [ ]:
# =============================================================
# CELL 9 — Move Post-Deploy Items into Workspace Folders
# =============================================================
# Moves the 7 items created by this notebook into the folders
# that Deploy_Bicycle_RTI Cell 5 already created.
# Safe to re-run — items already in correct folders are skipped.
#
# If folders don't exist yet (Deploy Cell 5 wasn't run), this cell
# creates them automatically.

import requests, json, time, re as _re
from datetime import datetime, timezone

ITEMS_API = f"{API}/workspaces/{ws_id}/items"
FOLDERS_API = f"{API}/workspaces/{ws_id}/folders"

# ── Helper functions ──

def get_or_create_folder(name):
    # Get existing folder by name or create it. Returns folder ID.
    r = requests.get(FOLDERS_API, headers=headers)
    if r.status_code == 200:
        for f in r.json().get("value", []):
            if f["displayName"] == name:
                return f["id"]
    r = requests.post(FOLDERS_API, headers=headers, json={"displayName": name})
    if r.status_code in (200, 201):
        return r.json()["id"]
    elif r.status_code == 409 or "AlreadyInUse" in r.text:
        r2 = requests.get(FOLDERS_API, headers=headers)
        for f in r2.json().get("value", []):
            if f["displayName"] == name:
                return f["id"]
    print(f"  [WARN] Could not create folder '{name}': {r.status_code}")
    return None

def move_item(item_id, item_name, folder_id, max_retries=3):
    # Move an item into a folder with retry on 429 rate limiting.
    for attempt in range(max_retries + 1):
        r = requests.post(f"{ITEMS_API}/{item_id}/move",
                          headers=headers,
                          json={"targetFolderId": folder_id})
        if r.status_code == 200:
            return "moved"
        elif r.status_code == 400 and "already" in r.text.lower():
            return "moved"  # already in folder
        elif r.status_code == 429:
            retry_after = 10 * (2 ** attempt)
            try:
                match = _re.search(r'until:\s*([\d/]+\s+[\d:]+\s+\w+)', r.text)
                if match:
                    blocked_until = datetime.strptime(match.group(1), "%m/%d/%Y %I:%M:%S %p")
                    blocked_until = blocked_until.replace(tzinfo=timezone.utc)
                    wait_secs = max(1, (blocked_until - datetime.now(timezone.utc)).total_seconds() + 2)
                    retry_after = min(wait_secs, 120)
            except Exception:
                pass
            if attempt < max_retries:
                print(f"    ⏳ {item_name} — rate limited, retrying in {int(retry_after)}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(retry_after)
            else:
                print(f"    ⚠️  {item_name} — rate limited after {max_retries} retries")
                return "failed"
        else:
            print(f"    ⚠️  {item_name}: {r.status_code} {r.text[:150]}")
            return "failed"
    return "failed"

# ── Post-Deploy items → folder mapping ──
# Only the 7 items created by THIS notebook. Deploy Cell 5 handles the other 21.
POST_DEPLOY_FOLDER_MAP = {
    "Ontology": [
        "Bicycle_Ontology_Model_New",
        "Bicycle_Ontology_Model_New_graph",
    ],
    "Agents": [
        "Bicycle Fleet Intelligence Agent",
        "ontology data agent",
        "Cycling-Campaign-Agent",
    ],
    "Activators": [
        "BicycleFleet_Activator",
        "Cycling Campaign Activator",
    ],
}

# ── Build item lookup ──
print("Organizing Post-Deploy items into folders...\n")
all_items_resp = requests.get(f"{ITEMS_API}?maxResults=500", headers=headers)
all_items = all_items_resp.json().get("value", []) if all_items_resp.status_code == 200 else []
item_lookup = {i["displayName"]: i["id"] for i in all_items}

# ── Execute moves ──
moved = 0
skipped = 0
not_found = 0

for folder_name, item_names in POST_DEPLOY_FOLDER_MAP.items():
    folder_id = get_or_create_folder(folder_name)
    if not folder_id:
        continue
    print(f"📁 {folder_name}")

    for name in item_names:
        item_id = item_lookup.get(name)
        if not item_id:
            print(f"    ⊘ {name} — not found in workspace")
            not_found += 1
            continue

        result = move_item(item_id, name, folder_id)
        if result == "moved":
            print(f"    ✅ {name}")
            moved += 1
        else:
            skipped += 1

        time.sleep(1.5)

print(f"\n✅ Post-Deploy folder organization complete: {moved} moved, {skipped} failed, {not_found} not found")
